# 13 — Granularity

## Which pruning choices preserve useful residues?

Notebook 07 compared reset and revision.

Notebook 13 asks a more specific question:

> Does the granularity of pruning change what survives?

This notebook is designed to run from a fresh Colab session.

It creates:

```text
results/13_granularity.csv
figures/13_granularity.png
```

In [ ]:
# Cell 1 — locate or clone repo

from pathlib import Path
import os
import sys
import subprocess

REPO_URL = "https://github.com/thinkthoughts/pruning-rml.git"
REPO_NAME = "pruning-rml"

cwd = Path.cwd().resolve()

if cwd.name == REPO_NAME:
    repo = cwd
elif cwd.name == "notebooks" and cwd.parent.name == REPO_NAME:
    repo = cwd.parent
elif (cwd / REPO_NAME).exists():
    repo = cwd / REPO_NAME
elif Path("/content/pruning-rml").exists():
    repo = Path("/content/pruning-rml")
else:
    target = Path("/content") if Path("/content").exists() else cwd
    os.chdir(target)
    subprocess.run(["git", "clone", REPO_URL], check=True)
    repo = target / REPO_NAME

os.chdir(repo)

src = repo / "src"
if str(src) not in sys.path:
    sys.path.insert(0, str(src))

print("repo:", repo)
print("src :", src)
print("src exists:", src.exists())

In [ ]:
# Cell 2 — ensure minimal src helpers

pkg = repo / "src" / "pruning_rml"
pkg.mkdir(parents=True, exist_ok=True)

(pkg / "__init__.py").write_text(
    '__version__ = "0.1.0"\n',
    encoding="utf-8",
)

(pkg / "paths.py").write_text(
    '''from pathlib import Path

def ensure_dirs(root, names=("results", "figures", "reports")):
    root = Path(root)
    outputs = {}
    for name in names:
        path = root / name
        path.mkdir(parents=True, exist_ok=True)
        outputs[name] = path
    return outputs
''',
    encoding="utf-8",
)

(pkg / "comparisons.py").write_text(
    '''def label_mode(method):
    method = method.lower()
    if "scratch" in method or "random" in method:
        return "reset"
    if "prun" in method or "revision" in method:
        return "revision"
    return "observe"
''',
    encoding="utf-8",
)

(pkg / "granularity.py").write_text(
    '''def granularity_score(granularity):
    granularity = granularity.lower()
    scores = {
        "depth": 1,
        "width": 2,
        "structured": 2,
        "semi-structured": 3,
        "sparse": 4,
        "fine-grained": 4,
    }
    return scores.get(granularity, 0)

def residue_question(granularity):
    granularity = granularity.lower()
    if granularity == "depth":
        return "Which layers are safe to remove?"
    if granularity == "width":
        return "Which channels, heads, or dimensions matter?"
    if granularity == "structured":
        return "Which larger components preserve capability?"
    if granularity in {"sparse", "fine-grained"}:
        return "Which individual weights or fine patterns survive?"
    return "What survives this pruning choice?"
''',
    encoding="utf-8",
)

for module_name in [
    "pruning_rml",
    "pruning_rml.paths",
    "pruning_rml.comparisons",
    "pruning_rml.granularity",
]:
    if module_name in sys.modules:
        del sys.modules[module_name]

print("src helpers ready")
print("package files:", sorted(p.name for p in pkg.iterdir()))

In [ ]:
# Cell 3 — imports and output folders

import pandas as pd
import matplotlib.pyplot as plt

from pruning_rml.paths import ensure_dirs
from pruning_rml.granularity import granularity_score, residue_question

outputs = ensure_dirs(repo)
results = outputs["results"]
figures = outputs["figures"]

print("results:", results)
print("figures:", figures)

## Granularity schema

This notebook does not claim benchmark reproduction.

It builds a comparison vocabulary for the paper's pruning families:

- **depth pruning** removes layers
- **width pruning** removes model dimensions, channels, or heads
- **structured pruning** removes larger identifiable components
- **sparse / fine-grained pruning** removes smaller weight-level structure

RML question:

> What kind of residue survives each pruning granularity?

In [ ]:
# Cell 4 — build granularity table

rows = [
    {
        "granularity": "depth",
        "what_is_removed": "layers",
        "revision_type": "coarse revision",
        "risk": "removes entire stages of computation",
        "possible_advantage": "simple and easy to interpret",
    },
    {
        "granularity": "width",
        "what_is_removed": "channels, heads, or hidden dimensions",
        "revision_type": "structured revision",
        "risk": "changes representational capacity",
        "possible_advantage": "preserves sequence of layers",
    },
    {
        "granularity": "structured",
        "what_is_removed": "larger identifiable components",
        "revision_type": "component revision",
        "risk": "may discard useful subcircuits",
        "possible_advantage": "hardware and deployment friendly",
    },
    {
        "granularity": "sparse",
        "what_is_removed": "individual or fine-grained weights",
        "revision_type": "fine revision",
        "risk": "harder to interpret and implement efficiently",
        "possible_advantage": "can preserve more inherited structure",
    },
    {
        "granularity": "fine-grained",
        "what_is_removed": "small parameter-level patterns",
        "revision_type": "fine revision",
        "risk": "requires careful retraining and evaluation",
        "possible_advantage": "may retain advantage longer",
    },
]

df = pd.DataFrame(rows)
df["granularity_score"] = df["granularity"].apply(granularity_score)
df["residue_question"] = df["granularity"].apply(residue_question)

df

In [ ]:
# Cell 5 — save CSV

csv_path = results / "13_granularity.csv"
df.to_csv(csv_path, index=False)

print("saved:", csv_path)
print("exists:", csv_path.exists())

## Visual summary

The `granularity_score` is an orientation score:

- lower = coarser pruning
- higher = finer pruning

The working hypothesis is:

> finer revision may preserve more inherited structure, but it becomes harder to interpret and deploy.

In [ ]:
# Cell 6 — save figure

fig_path = figures / "13_granularity.png"

ax = df.plot.bar(
    x="granularity",
    y="granularity_score",
    legend=False,
    figsize=(9, 5),
)

ax.set_title("Pruning granularity as revision")
ax.set_xlabel("Granularity")
ax.set_ylabel("Fineness / residue-preservation score")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.savefig(fig_path, dpi=160)
plt.show()

print("saved:", fig_path)
print("exists:", fig_path.exists())

In [ ]:
# Cell 7 — verify outputs

expected = [
    results / "13_granularity.csv",
    figures / "13_granularity.png",
]

for path in expected:
    print(path, "exists:", path.exists())

## Notebook result

Notebook 13 creates:

```text
results/13_granularity.csv
figures/13_granularity.png
```

Next notebook:

```text
notebooks/17_revision_not_reset.ipynb
```

Next question:

> When does pruning behave like meaningful revision rather than destructive deletion?